# 01 · Data Exploration

Explore the JWST f150w dataset: class distributions, effective radius, delta confidence,
axis ratio, and sample images — all without loading the model.

**Sections:**
1. Filter configuration
2. Load metadata from H5 files
3. Distribution statistics (per morphology class)
4. Sample image gallery
5. Patch tokenization coverage

In [ ]:
import os, sys, glob
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm
from torchvision import transforms

PROJECT_ROOT = "/u/yacheng/projects/ssl_outthere"
sys.path.insert(0, PROJECT_ROOT)
from encoder_image.astrodino.train.data.augmentations import ToRGB

PIXEL_SCALE_MAS = 30        # mas per pixel
DEG_TO_PIXEL    = 3600 * 1000 / PIXEL_SCALE_MAS   # 120000 px/deg

MORPH_NAMES   = {0: 'Spheroid', 1: 'Disk-dom.', 2: 'Irregular', 3: 'Bulge-dom.'}
MORPH_COLORS  = {0: '#E74C3C', 1: '#3498DB',   2: '#2ECC71',   3: '#9B59B6'}
print('Imports OK')

## 1 · Filter Configuration
Change the values below and re-run from this cell to explore different subsets.

In [ ]:
DATA_ROOT = "/u/yacheng/projects/ssl_outthere/images/jwst/f150w"

# ── FILTER OPTIONS ──────────────────────────────────────────────────────────
DELTA_THRESHOLD    = 0.5    # max delta_f150w (classification confidence); None = no filter
REFF_MIN_PIX       = 2.0    # min effective radius [pixels];  None = no lower bound
REFF_MAX_PIX       = None   # max effective radius [pixels];  None = no upper bound
EXCLUDE_IRREGULAR  = False  # True = drop morph class 2 (Irregular) from all plots
# ────────────────────────────────────────────────────────────────────────────

CROP_SIZE        = 64       # display crop size
N_EXAMPLES       = 8        # images per class in gallery
SEED             = 42
rng = np.random.default_rng(SEED)
print(f'Filters: delta<{DELTA_THRESHOLD}, reff∈[{REFF_MIN_PIX}, {REFF_MAX_PIX}] px, exclude_irr={EXCLUDE_IRREGULAR}')

## 2 · Load Metadata

In [ ]:
h5_files = sorted(glob.glob(os.path.join(DATA_ROOT, '*.h5')))
print(f'Found {len(h5_files)} h5 files')

records = []   # list of dicts: one per valid sample

for fpath in tqdm(h5_files, desc='Loading metadata'):
    with h5py.File(fpath, 'r') as f:
        n = len(f['image'])
        morph   = f['morph_flag_f150w'][:] if 'morph_flag_f150w' in f else np.full(n, np.nan)
        delta   = f['delta_f150w'][:]      if 'delta_f150w'      in f else np.full(n, np.nan)
        re_deg  = f['radius_sersic'][:]    if 'radius_sersic'    in f else np.full(n, np.nan)
        ar      = f['axis_ratio'][:]       if 'axis_ratio'       in f else np.full(n, np.nan)

        re_pix = re_deg * DEG_TO_PIXEL

        for i in range(n):
            records.append({
                'fpath': fpath, 'idx': i,
                'morph': morph[i], 'delta': delta[i],
                're_pix': re_pix[i], 'axis_ratio': ar[i]
            })

print(f'Total samples loaded: {len(records)}')

In [ ]:
import numpy as np

morph_arr   = np.array([r['morph']      for r in records], dtype=float)
delta_arr   = np.array([r['delta']      for r in records], dtype=float)
re_arr      = np.array([r['re_pix']     for r in records], dtype=float)
ar_arr      = np.array([r['axis_ratio'] for r in records], dtype=float)

# Base mask: valid morph & delta
mask = np.isfinite(morph_arr) & np.isfinite(delta_arr)

# Apply user filters
if DELTA_THRESHOLD is not None:
    mask &= (delta_arr < DELTA_THRESHOLD)
if REFF_MIN_PIX is not None:
    mask &= np.isfinite(re_arr) & (re_arr >= REFF_MIN_PIX)
if REFF_MAX_PIX is not None:
    mask &= np.isfinite(re_arr) & (re_arr <= REFF_MAX_PIX)
if EXCLUDE_IRREGULAR:
    mask &= (morph_arr != 2)

filtered = [r for r, m in zip(records, mask) if m]
morph_f  = morph_arr[mask]
delta_f  = delta_arr[mask]
re_f     = re_arr[mask]
ar_f     = ar_arr[mask]

# Only show classes that remain after filtering
classes = sorted(int(c) for c in np.unique(morph_f) if np.isfinite(c))

print(f'After filtering: {len(filtered)} samples (from {len(records)})')
if EXCLUDE_IRREGULAR:
    print('  ⚠ Irregular galaxies (class 2) excluded')
print('\nClass distribution:')
unique, counts = np.unique(morph_f.astype(int), return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {MORPH_NAMES.get(u, u):15s}  {c:5d}  ({100*c/len(filtered):.1f}%)')

## 3 · Distribution Statistics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
# `classes` is set dynamically in the apply-filters cell (respects EXCLUDE_IRREGULAR)

# ── Row 0: delta distribution ──────────────────────────────────────────────
ax = axes[0, 0]
ax.hist(delta_f, bins=50, color='gray', edgecolor='black', alpha=0.7)
ax.set_xlabel('delta (classification uncertainty)')
ax.set_ylabel('Count')
ax.set_title('All classes: delta distribution')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for c in classes:
    sel = delta_f[morph_f == c]
    if len(sel): ax.hist(sel, bins=40, alpha=0.5, label=MORPH_NAMES[c], color=MORPH_COLORS[c])
ax.set_xlabel('delta')
ax.set_title('Delta by morphology class')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[0, 2]
for c in classes:
    sel = delta_f[morph_f == c]
    if len(sel):
        sorted_d = np.sort(sel)
        ax.plot(sorted_d, np.linspace(0, 1, len(sel)), label=MORPH_NAMES[c], color=MORPH_COLORS[c])
ax.set_xlabel('delta')
ax.set_ylabel('CDF')
ax.set_title('Delta CDF by class')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Row 1: effective radius distribution ──────────────────────────────────
re_valid = re_f[np.isfinite(re_f) & (re_f > 0)]
log_re   = np.log10(re_valid)

ax = axes[1, 0]
ax.hist(log_re, bins=60, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(np.mean(log_re), color='red',    linestyle='--', label=f'mean={np.mean(log_re):.2f}')
ax.axvline(np.median(log_re), color='orange', linestyle='--', label=f'median={np.median(log_re):.2f}')
ax.set_xlabel('log₁₀(r_eff [pixels])')
ax.set_title('All classes: r_eff distribution')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
for c in classes:
    sel = re_f[(morph_f == c) & np.isfinite(re_f) & (re_f > 0)]
    if len(sel): ax.hist(np.log10(sel), bins=40, alpha=0.5, label=MORPH_NAMES[c], color=MORPH_COLORS[c])
ax.set_xlabel('log₁₀(r_eff [pixels])')
ax.set_title('r_eff by morphology class')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1, 2]
for c in classes:
    sel = re_f[(morph_f == c) & np.isfinite(re_f) & (re_f > 0)]
    if len(sel):
        sorted_re = np.sort(np.log10(sel))
        ax.plot(sorted_re, np.linspace(0, 1, len(sel)), label=MORPH_NAMES[c], color=MORPH_COLORS[c])
ax.set_xlabel('log₁₀(r_eff [pixels])')
ax.set_ylabel('CDF')
ax.set_title('r_eff CDF by class')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

irr_note = '  excl. Irregular' if EXCLUDE_IRREGULAR else ''
plt.suptitle(f'Dataset statistics  (delta<{DELTA_THRESHOLD}, reff∈[{REFF_MIN_PIX}, {REFF_MAX_PIX}] px, N={len(filtered)}{irr_note})',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Axis ratio distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ar_valid = ar_f[np.isfinite(ar_f)]
ax.hist(ar_valid, bins=50, color='coral', edgecolor='black', alpha=0.7)
ax.set_xlabel('Axis ratio (b/a)')
ax.set_title('All classes: axis ratio distribution')
ax.grid(True, alpha=0.3)

ax = axes[1]
for c in classes:
    sel = ar_f[(morph_f == c) & np.isfinite(ar_f)]
    if len(sel): ax.hist(sel, bins=40, alpha=0.5, label=MORPH_NAMES[c], color=MORPH_COLORS[c])
ax.set_xlabel('Axis ratio (b/a)')
ax.set_title('Axis ratio by class')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 2D: r_eff vs axis_ratio scatter
ax = axes[2]
for c in classes:
    sel_re = re_f[(morph_f == c) & np.isfinite(re_f) & (re_f > 0) & np.isfinite(ar_f)]
    sel_ar = ar_f[(morph_f == c) & np.isfinite(re_f) & (re_f > 0) & np.isfinite(ar_f)]
    if len(sel_re):
        sub = rng.choice(len(sel_re), size=min(500, len(sel_re)), replace=False)
        ax.scatter(np.log10(sel_re[sub]), sel_ar[sub], s=4, alpha=0.4,
                   label=MORPH_NAMES[c], color=MORPH_COLORS[c])
ax.set_xlabel('log₁₀(r_eff [pixels])')
ax.set_ylabel('Axis ratio (b/a)')
ax.set_title('r_eff vs axis ratio')
ax.legend(fontsize=8, markerscale=3)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Per-class summary table
print(f"{'Class':<15} {'N':>6}  {'delta mean':>10} {'re_pix med':>10} {'ar mean':>8}")
print('-' * 58)
for c in classes:
    sel_mask = morph_f == c
    if not sel_mask.any(): continue
    n_c    = sel_mask.sum()
    d_mean = np.nanmean(delta_f[sel_mask])
    re_med = np.nanmedian(re_f[sel_mask])
    ar_m   = np.nanmean(ar_f[sel_mask])
    print(f'{MORPH_NAMES[c]:<15} {n_c:>6}  {d_mean:>10.3f} {re_med:>10.2f} {ar_m:>8.3f}')

## 4 · Sample Image Gallery

In [ ]:
to_rgb = ToRGB()
center_crop = transforms.CenterCrop(CROP_SIZE)

def load_image(record):
    with h5py.File(record['fpath'], 'r') as f:
        img = f['image'][record['idx']].astype('float32')
    import torch
    img3 = np.repeat(img[np.newaxis], 3, axis=0)
    import torch
    t = center_crop(torch.from_numpy(img3))
    t = to_rgb(t.numpy())
    return np.transpose(t, (1, 2, 0))   # HWC for imshow

In [ ]:
fig, axes = plt.subplots(len(classes), N_EXAMPLES, figsize=(N_EXAMPLES * 1.8, len(classes) * 1.8))
fig.suptitle(f'Sample images by morphology class  (delta<{DELTA_THRESHOLD}, reff>={REFF_MIN_PIX} px)',
             fontsize=12)

for row, c in enumerate(classes):
    class_samples = [r for r, m in zip(filtered, [morph_f[i] == c for i in range(len(filtered))]) if m]
    # rebuild per-class lists directly
    class_records = [r for r in filtered if r['morph'] == c]
    chosen = rng.choice(len(class_records), size=min(N_EXAMPLES, len(class_records)), replace=False)

    for col in range(N_EXAMPLES):
        ax = axes[row, col]
        ax.axis('off')
        if col < len(chosen):
            rec = class_records[chosen[col]]
            try:
                img = load_image(rec)
                ax.imshow(img)
                ax.set_title(f"δ={rec['delta']:.2f}\nre={rec['re_pix']:.1f}px",
                             fontsize=6, color='white',
                             bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.6))
            except Exception as e:
                ax.text(0.5, 0.5, str(e), ha='center', va='center', fontsize=6, transform=ax.transAxes)
        if col == 0:
            ax.set_ylabel(MORPH_NAMES[c], fontsize=9, color=MORPH_COLORS[c], rotation=90,
                          labelpad=4)
            ax.axis('on')
            ax.set_yticks([]); ax.set_xticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor(MORPH_COLORS[c]); spine.set_linewidth(2)

plt.tight_layout()
plt.show()

## 5 · Patch Tokenization Coverage

How many patches (of a given size) fall within `N × r_eff` of the galaxy centre?

In [ ]:
PATCH_SIZES = [4, 8, 14, 16, 32]   # patch edge lengths in pixels
REFF_RADIUS = 2                     # count patches within this many r_eff
IMG_SIZE    = 128                   # native image size

# Use filtered samples with valid r_eff
valid_re = re_f[np.isfinite(re_f) & (re_f > 0)]
valid_morph_for_patch = morph_f[np.isfinite(re_f) & (re_f > 0)]

fig, axes = plt.subplots(1, len(PATCH_SIZES), figsize=(4 * len(PATCH_SIZES), 4), sharey=False)

for ax, ps in zip(axes, PATCH_SIZES):
    # Patches on a grid: top-left corners at multiples of ps, centred on image
    centre = IMG_SIZE / 2.0
    n_patches_per_galaxy = []
    for re in valid_re:
        radius = REFF_RADIUS * re   # pixel radius
        count = 0
        for py in range(0, IMG_SIZE, ps):
            for px in range(0, IMG_SIZE, ps):
                cx, cy = px + ps / 2, py + ps / 2   # patch centre
                if np.hypot(cx - centre, cy - centre) <= radius:
                    count += 1
        n_patches_per_galaxy.append(count)
    n_patches = np.array(n_patches_per_galaxy)

    for c in classes:
        sel = n_patches[valid_morph_for_patch == c]
        if len(sel):
            ax.hist(sel, bins=range(0, int(n_patches.max()) + 2),
                    alpha=0.5, label=MORPH_NAMES[c], color=MORPH_COLORS[c])

    ax.set_xlabel('# patches inside 2×r_eff')
    ax.set_title(f'patch={ps}px')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Count')
plt.suptitle(f'Patches within {REFF_RADIUS}×r_eff by patch size  (N={len(valid_re)})', fontsize=12)
plt.tight_layout()
plt.show()